<h1>Recycling classifier (CNN)</h1>
<h3>This project uses the <a href="https://www.kaggle.com/datasets/zlatan599/garbage-dataset-classification"> Garbage Image dataset on Kaggle</a></h3>

In [ ]:
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.utils import to_categorical
from tensorflow.keras import backend as K
from tensorflow.keras.applications import ResNet50 
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.models import load_model
from tensorflow.keras import layers, models
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
import pandas as pd
import os

<h4>Load path from environmental variables and ensure all is working with print statements</h4>

In [ ]:
path = os.environ["DATASET_DIR"]

if path is None:
    raise EnvironmentError(
        "DATASET_DIR environment variable not set.\n"
        "Please set it before running this script:\n"
        "  export DATASET_DIR=/path/to/garbage-dataset"
    )

print(f"Using dataset directory: {path}")

In [ ]:
os.environ.get("DATASET_DIR")

In [ ]:
metadata = pd.read_csv(os.path.join(path, "metadata.csv"))

images_dir = os.path.join(path, "images")

classes = {
    cls: os.listdir(os.path.join(images_dir, cls))
    for cls in sorted(os.listdir(images_dir))
    if os.path.isdir(os.path.join(images_dir, cls))
}

print(metadata.head())
print(classes.keys())

<h4>Divide dataset into metadata (Labels) and a dictionary with the structure - {Class: Image1, Image2, Image3}</h4>

In [ ]:
x = []
y = []

class_names = list(classes.keys())
print(class_names)
class_to_idx = {c: i for i, c in enumerate(class_names)}
print(class_to_idx)

for label, files in classes.items():
    for file in files:
        try:
            img_path = os.path.join(path,'images', label, file)
            img = load_img(img_path, target_size=(224,224), color_mode="rgb")

            img_array = img_to_array(img)
            x.append(img_array)
            y.append(class_to_idx[label])
        except Exception as e:
            print(f"Failed to load {file}: {e}")

print(len(x), len(y))
x = np.array(x, dtype="float32")
y = to_categorical(np.array(y), num_classes = len(class_names))

print("Dataset shape:", x.shape, y.shape)

In [ ]:
y_labels = np.array(y)

x_train, x_temp, y_train, y_temp = train_test_split(
    x, y, test_size=0.2, random_state=42, stratify=y_labels
)

y_temp_indices = np.argmax(y_temp, axis=1)

x_val, x_test, y_val, y_test = train_test_split(
    x_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp_indices
)

print("Train:", x_train.shape, y_train.shape)
print("Val:", x_val.shape, y_val.shape)
print("Test:", x_test.shape, y_test.shape)

In [ ]:
K.clear_session()

print("="*10)
print("Training Phase 1")
print("="*10)

data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
    tf.keras.layers.RandomBrightness(0.2),
    tf.keras.layers.RandomContrast(0.2),
])

base_model = ResNet50(
    include_top=False,
    weights="imagenet",
    input_shape=(224, 224, 3)
)

base_model.trainable = False

early_stopping = EarlyStopping(
    monitor = "val_loss",
    patience=5,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=2,
    verbose=1,
)

model = models.Sequential([
    data_augmentation,
    layers.Lambda(preprocess_input),
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.BatchNormalization(),
    layers.Dropout(0.4),
    layers.Dense(512, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.5),
    layers.Dense(6, activation='softmax')

])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=3e-4),
    loss="categorical_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.TopKCategoricalAccuracy(k=2, name="top2_acc")
             ],
)

# First Round Training
history = model.fit(x_train, y_train, 
                    epochs=10, 
                    batch_size=32, 
                    validation_data=(x_val, y_val),
                    callbacks=[early_stopping, reduce_lr],
                    verbose=1)

In [ ]:
models_dir = os.path.join(path, "models")
os.makedirs(models_dir, exist_ok=True)

model_path = os.path.join(models_dir, "base_modelv1.keras")
model.save(model_path)

print(f"Model saved to: {model_path}")

<h4>Uncomment to use any pretrained base model<h4>

In [ ]:
# base_model = load_model(
#     os.path.join(models_dir, "base_modelv1.keras"),
# )

"""
Use line 'safe_mode=False' before end bracket if base_model is trusted and error is thrown
"""

In [ ]:
print("="*10)
print("Training Phase 2")
print("="*10)

for layer in base_model.layers[:-4]:
    layer.trainable = False

for layer in base_model.layers[-4:]:
    layer.trainable = True


model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="categorical_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.TopKCategoricalAccuracy(k=2, name="top2_acc")
        ],
)

# Second Round Training
history_fine = model.fit(x_train, y_train, 
                    epochs=20, 
                    batch_size=32, 
                    validation_data=(x_val, y_val),
                    callbacks=[early_stopping, reduce_lr],
                    verbose=1)

In [ ]:
"""
### Custom model - 75-80% accurcay across all metrics ###

model = tf.keras.models.Sequential([
    data_augmentation,

    tf.keras.layers.Conv2D(32, (3,3), activation="relu", input_shape=(224,224,3)),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D((2,2)),

    tf.keras.layers.Conv2D(64, (3,3), activation="relu"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D((2,2)),

    tf.keras.layers.Conv2D(128, (3,3), activation="relu"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D((2,2)),

    tf.keras.layers.Conv2D(256, (3,3), activation="relu"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D((2,2)),

    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dropout(0.4),
    tf.keras.layers.Dense(256, activation="relu"),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(6, activation="softmax")
])
"""

In [ ]:
plt.plot(history.history["val_accuracy"] + history_fine.history["val_accuracy"])

In [ ]:
# Combine histories
def combine_histories(h1, h2):
    history_combined = {}
    for key in h1.history.keys():
        history_combined[key] = h1.history[key] + h2.history[key]
    return history_combined

combined_history = combine_histories(history, history_fine)

# Plot combined training
plt.figure(figsize=(14, 5))

# Accuracy plot
plt.subplot(1, 2, 1)
plt.plot(combined_history["accuracy"], label="Train Accuracy")
plt.plot(combined_history["val_accuracy"], label="Validation Accuracy")
plt.axvline(x=len(history.history["accuracy"]), color='r', 
            linestyle='--', label='Start Fine-tuning')
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.legend()
plt.title("Training vs Validation Accuracy (Both Phases)")

# Loss plot
plt.subplot(1, 2, 2)
plt.plot(combined_history["loss"], label="Train Loss")
plt.plot(combined_history["val_loss"], label="Validation Loss")
plt.axvline(x=len(history.history["loss"]), color='r', 
            linestyle='--', label='Start Fine-tuning')
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()
plt.title("Training vs Validation Loss (Both Phases)")

plt.tight_layout()
plt.show()

In [ ]:
y_pred_probs = model.predict(x_test)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = np.argmax(y_test, axis=1)

print(classification_report(y_true, y_pred, target_names=class_names))

In [ ]:
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", 
            xticklabels=class_names, 
            yticklabels=class_names)

plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
plt.show()

In [ ]:
test_loss, test_acc, test_top2 = model.evaluate(x_test, y_test)
print(f"Test accuracy: {test_acc * 100:.2f}%")
print(f"Top-2 accuracy: {test_top2 * 100:.2f}%")
print(f"Test loss: {test_loss:.4f}")

In [ ]:
final_model_path = os.path.join(models_dir, "final_modelv1.keras")
model.save(final_model_path)

print(f"Model saved to: {final_model_path}")